In [ ]:
!pip install -q "torchao>=0.16.0"
!pip install -q transformers bitsandbytes trl peft datasets

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from tqdm import tqdm

In [ ]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
print("Модель загружена!")

In [ ]:
# Проверка токенизатора
tokens = tokenizer.encode("Привет, Квен!")
print("Токенизатор работает:", tokenizer.decode(tokens))

# Тестовая генерация
messages = [
    {"role": "system", "content": "Ты — продвинутый ИИ-ассистент. Отвечаешь четко, развернуто и только на русском языке."},
    {"role": "user", "content": "Скажи hello world!"},
]
model_inputs = tokenizer(
    [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)],
    return_tensors="pt"
).to(model.device)

generated_ids = model.generate(**model_inputs, max_new_tokens=128, temperature=0.7, do_sample=True, top_p=0.9)
generated_ids = [out[len(inp):] for inp, out in zip(model_inputs.input_ids, generated_ids)]
print(tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0])

In [ ]:
# d0rj/ru-instruct имеет колонки: context (system), instruction (user), response (assistant)
def format_example(example):
    messages = []
    if example.get("context"):
        messages.append({"role": "system", "content": example["context"]})
    if example.get("instruction"):
        messages.append({"role": "user", "content": example["instruction"]})
    if example.get("response"):
        messages.append({"role": "assistant", "content": example["response"]})
    if messages:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return ""


In [ ]:
dataset = load_dataset("d0rj/ru-instruct", split="train[:5000]")

peft_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)

training_args = SFTConfig(
    output_dir="./results_fine_tuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=5,
    save_strategy="steps",
    save_steps=50,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_args,
    formatting_func=format_example,
)

print("Начало обучения...")
trainer.train()
print("Готово!")

In [ ]:
trainer.model.save_pretrained("my_custom_chat_bot")
tokenizer.save_pretrained("my_custom_chat_bot")
print("Веса сохранены в my_custom_chat_bot/")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print("Загрузка базовой модели...")
base_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map="auto")

print("Загрузка дообученной модели (база + LoRA)...")
finetuned_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, device_map="auto")
finetuned_model = PeftModel.from_pretrained(finetuned_model, "my_custom_chat_bot")
finetuned_model.eval()

print("Обе модели загружены!")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Тестовая выборка — примеры, не участвовавшие в обучении
test_dataset = load_dataset("d0rj/ru-instruct", split="train[5000:5500]")
test_dataset = test_dataset.map(lambda x: {"text": format_example(x)})
test_dataset = test_dataset.filter(lambda x: x["text"] != "")
print(f"Тестовых примеров: {len(test_dataset)}")

def calculate_perplexity(model, dataset, max_length=1024):
    model.eval()
    nlls = []
    with torch.no_grad():
        for example in tqdm(dataset):
            ids = tokenizer(
                example["text"], return_tensors="pt", truncation=True, max_length=max_length
            ).input_ids.to(device)
            loss = model(ids, labels=ids.clone()).loss
            nlls.append(loss)
    return torch.exp(torch.stack(nlls).mean()).item()

base_ppl      = calculate_perplexity(base_model, test_dataset)
finetuned_ppl = calculate_perplexity(finetuned_model, test_dataset)

print(f"\n=== СРАВНЕНИЕ ПЕРПЛЕКСИИ ===")
print(f"Базовая модель:     {base_ppl:.4f}")
print(f"Дообученная модель: {finetuned_ppl:.4f}")
print(f"Улучшение:          {base_ppl - finetuned_ppl:.4f}  (ниже = лучше)")